# **Boolean Information Retrieval**

# All Imports Here

In [95]:
import os
from pathlib import Path

src_dir = Path.cwd().parent
os.chdir(src_dir)

import pandas as pd
from data_preprocessor.preprocessor import DataPreprocessor
from collections import defaultdict


# Function to Add Documents

In [96]:
def add_docs(body: list):
    doc_dict = {}
    for i, text in enumerate(body):
        doc_dict[f"doc{i+1}"] = text
    return doc_dict


# Function to Create Inverted Index

In [97]:
def build_inverted_index(df):
    index = defaultdict(set)

    for _, row in df.iterrows():
        for term in row["body_clean"]:
            index[term].add(row["documents"])

    return dict(index)

# Boolean Retrieval Function

In [98]:

def boolean_retrieval(query, index, all_documents):
    tokens = query.lower().split()

    def postings(term):
        return set(index.get(term, set()))

    operands = []
    operators = []

    i = 0

    while i < len(tokens):
        token = tokens[i]

        if token == "not":
            if i + 1 >= len(tokens):
                raise ValueError("NOT must be followed by a term")

            operands.append(
                set(all_documents) - postings(tokens[i + 1])
            )
            i += 2

        elif token in {"and", "or"}:
            operators.append(token)
            i += 1

        else:
            operands.append(postings(token))
            i += 1

    if not operands:
        return []

    # evaluate AND first
    new_operands = [operands[0]]
    new_operators = []

    operand_idx = 1

    for op in operators:
        if op == "and":
            new_operands[-1] &= operands[operand_idx]
        else:
            new_operators.append(op)
            new_operands.append(operands[operand_idx])

        operand_idx += 1

    # then evaluate OR
    result = new_operands[0]

    for operand in new_operands[1:]:
        result |= operand

    return sorted(result)

# Retrieval In Action

In [99]:
# pass doc body
body = [
    'Today, the Department announced a BRIGHT new policy!!! Please visit the official page for more details and updates.',
    'Pls visit the company link: www.example.com. The staff will HELP employees grow, learn, and improve their skills.',
    'The Rose Project is SHINING this year!!! For more information, please contact the project team via email.',
    'The Finance Department approved the annual BUDGET yesterday. Employees should review the updated financial guidelines carefully.',
    'Management will conduct a TRAINING session on Friday!!! All staff members are requested to attend the program.',
    'The company launched a new DIGITAL project to improve customer service and increase overall efficiency.',
    'Employees are encouraged to submit their REPORTS before Monday. Late submissions may affect the project review process.',
    'The Human Resources team announced a new recruitment POLICY!!! Please check the careers page for available positions.'
]

docs = add_docs(body)

df = pd.DataFrame()
df['documents'], df['body'] = docs.keys(), docs.values()
df.head()

,documents,body
0,doc1,"Today, the Department announced a BRIGHT new p..."
1,doc2,Pls visit the company link: www.example.com. T...
2,doc3,The Rose Project is SHINING this year!!! For m...
3,doc4,The Finance Department approved the annual BUD...
4,doc5,Management will conduct a TRAINING session on ...


In [100]:
# clean the body texts
dp = DataPreprocessor(df, ['body'])
df_clean = dp.run_preprocessor()
df_clean['body_combined'] = dp.combine_text()
df_clean['documents'] = df['documents'].copy()
df_clean['body'] = df['body'].copy()
df_clean.head()

,body,body_clean,body_combined,documents
0,"Today, the Department announced a BRIGHT new p...","[today, department, announced, bright, new, po...",today department announced bright new policy p...,doc1
1,Pls visit the company link: www.example.com. T...,"[pls, visit, company, link, www, example, com,...",pls visit company link www example com staff h...,doc2
2,The Rose Project is SHINING this year!!! For m...,"[rose, project, shining, year, information, pl...",rose project shining year information please c...,doc3
3,The Finance Department approved the annual BUD...,"[finance, department, approved, annual, budget...",finance department approved annual budget yest...,doc4
4,Management will conduct a TRAINING session on ...,"[management, conduct, training, session, frida...",management conduct training session friday sta...,doc5


In [101]:
# build inverted index
inverted_index = build_inverted_index(df_clean)
all_documents = set(df_clean["documents"])

print("Inverted Index is: ")
inverted_index

Inverted Index is: 


{'today': {'doc1'},
 'department': {'doc1', 'doc4'},
 'announced': {'doc1', 'doc8'},
 'bright': {'doc1'},
 'new': {'doc1', 'doc6', 'doc8'},
 'policy': {'doc1', 'doc8'},
 'please': {'doc1', 'doc3', 'doc8'},
 'visit': {'doc1', 'doc2'},
 'official': {'doc1'},
 'page': {'doc1', 'doc8'},
 'detail': {'doc1'},
 'update': {'doc1'},
 'pls': {'doc2'},
 'company': {'doc2', 'doc6'},
 'link': {'doc2'},
 'www': {'doc2'},
 'example': {'doc2'},
 'com': {'doc2'},
 'staff': {'doc2', 'doc5'},
 'help': {'doc2'},
 'employee': {'doc2', 'doc4', 'doc7'},
 'grow': {'doc2'},
 'learn': {'doc2'},
 'improve': {'doc2', 'doc6'},
 'skill': {'doc2'},
 'rose': {'doc3'},
 'project': {'doc3', 'doc6', 'doc7'},
 'shining': {'doc3'},
 'year': {'doc3'},
 'information': {'doc3'},
 'contact': {'doc3'},
 'team': {'doc3', 'doc8'},
 'via': {'doc3'},
 'email': {'doc3'},
 'finance': {'doc4'},
 'approved': {'doc4'},
 'annual': {'doc4'},
 'budget': {'doc4'},
 'yesterday': {'doc4'},
 'review': {'doc4', 'doc7'},
 'updated': {'doc4'},
 

In [ ]:
# search using query
query = "policy OR department"

result = boolean_retrieval(
    query,
    inverted_index,
    all_documents
)
print(f"QUERY: {query}\nResult: ")
for doc in result:
    body = df_clean.loc[
        df_clean['documents'] == doc,
        "body"
    ].iloc[0]
    print(f"{doc}: {body}")

QUERY: policy OR department
Result: 
doc1: Today, the Department announced a BRIGHT new policy!!! Please visit the official page for more details and updates.
doc4: The Finance Department approved the annual BUDGET yesterday. Employees should review the updated financial guidelines carefully.
doc8: The Human Resources team announced a new recruitment POLICY!!! Please check the careers page for available positions.
